# Model Evaluation and Cross-Validation: Wine Classification

## Introduction
This notebook is designed to show that machine learning competence includes evaluation discipline, not just model training. The project compares several models on the same dataset using both a train/test split and cross-validation.

## Project Goal
Compare candidate classifiers fairly, understand why one train/test split is not enough, and justify model choice using both mean performance and stability.

## Machine Learning Concepts Used
- Model Comparison
- Train/Test Split
- Cross-Validation
- Accuracy
- Macro F1
- Variance Across Folds
- Feature Subset Refinement

## Dataset
`sklearn.datasets.load_wine`

## Step 1: Import libraries

**What this section is doing**  
Import the data, models, and evaluation tools needed for a fair multi-model comparison.

**Why this matters**  
This section exists so the workflow stays explicit, interpretable, and reproducible. In a professional notebook, each code cell should have a clear purpose before it is executed.

In [1]:
import pandas as pd

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

## Step 2: Load the dataset

**What this section is doing**  
Use the wine dataset because it is clean, multi-class, and compact enough for repeated experiments.

**Why this matters**  
This section exists so the workflow stays explicit, interpretable, and reproducible. In a professional notebook, each code cell should have a clear purpose before it is executed.

In [2]:
data = load_wine(as_frame=True)
X = data.data.copy()
y = data.target.copy()

print("Dataset shape:", X.shape)
display(X.head())

Dataset shape: (178, 13)


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0


## Step 3: Inspect feature relationships

**What this section is doing**  
Before comparing models, inspect which variables seem most related to the target. This creates context for the later refinement experiment.

**Why this matters**  
This section exists so the workflow stays explicit, interpretable, and reproducible. In a professional notebook, each code cell should have a clear purpose before it is executed.

In [3]:
corr_proxy = pd.concat([X, y.rename("target")], axis=1).corr(numeric_only=True)["target"].abs().sort_values(ascending=False)
display(corr_proxy.to_frame("absolute_correlation_with_target"))

,absolute_correlation_with_target
target,1.000000
flavanoids,0.847498
od280/od315_of_diluted_wines,0.788230
total_phenols,0.719163
proline,0.633717
hue,0.617369
alcalinity_of_ash,0.517859
proanthocyanins,0.499130
nonflavanoid_phenols,0.489109
malic_acid,0.437776


## Step 4: Define candidate models

**What this section is doing**  
Use a small but meaningful comparison set with different assumptions and complexity levels.

**Why this matters**  
This section exists so the workflow stays explicit, interpretable, and reproducible. In a professional notebook, each code cell should have a clear purpose before it is executed.

In [4]:
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "DecisionTreeClassifier": DecisionTreeClassifier(max_depth=5, random_state=42),
    "RandomForestClassifier": RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
}

models

{'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
 'DecisionTreeClassifier': DecisionTreeClassifier(max_depth=5, random_state=42),
 'RandomForestClassifier': RandomForestClassifier(max_depth=6, n_estimators=200, random_state=42)}

## Step 5: Train/test comparison

**What this section is doing**  
A train/test split gives an intuitive first comparison, but it should not be the only basis for selecting a model.

**Why this matters**  
This section exists so the workflow stays explicit, interpretable, and reproducible. In a professional notebook, each code cell should have a clear purpose before it is executed.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rows.append({
        "model": name,
        "test_accuracy": accuracy_score(y_test, preds),
        "test_macro_f1": f1_score(y_test, preds, average="macro")
    })

test_results = pd.DataFrame(rows).sort_values("test_macro_f1", ascending=False)
display(test_results)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,model,test_accuracy,test_macro_f1
2,RandomForestClassifier,1.000000,1.000000
0,LogisticRegression,0.972222,0.970962
1,DecisionTreeClassifier,0.944444,0.945741


## Step 6: Cross-validation comparison

**What this section is doing**  
Cross-validation produces a more stable estimate by repeating the evaluation across several folds.

**Why this matters**  
This section exists so the workflow stays explicit, interpretable, and reproducible. In a professional notebook, each code cell should have a clear purpose before it is executed.

In [6]:
cv_rows = []
for name, model in models.items():
    scores = cross_val_score(
        model, X, y,
        cv=5,
        scoring="f1_macro",
        n_jobs=-1
    )
    cv_rows.append({
        "model": name,
        "cv_macro_f1_mean": scores.mean(),
        "cv_macro_f1_std": scores.std()
    })

cv_results = pd.DataFrame(cv_rows).sort_values("cv_macro_f1_mean", ascending=False)
display(cv_results)

,model,cv_macro_f1_mean,cv_macro_f1_std
2,RandomForestClassifier,0.967684,0.020086
0,LogisticRegression,0.962064,0.032699
1,DecisionTreeClassifier,0.865368,0.047314


## Step 7: Refine the comparison on a smaller feature set

**What this section is doing**  
Repeat the evaluation on a smaller set of stronger features to see whether the ranking of models changes.

**Why this matters**  
This section exists so the workflow stays explicit, interpretable, and reproducible. In a professional notebook, each code cell should have a clear purpose before it is executed.

In [7]:
selected_features = corr_proxy.drop("target").head(6).index.tolist()
print("Selected features:", selected_features)

X_small = X[selected_features]

small_rows = []
for name, model in models.items():
    scores = cross_val_score(
        model, X_small, y,
        cv=5,
        scoring="f1_macro",
        n_jobs=-1
    )
    small_rows.append({
        "model": name,
        "selected_feature_cv_macro_f1_mean": scores.mean(),
        "selected_feature_cv_macro_f1_std": scores.std()
    })

small_results = pd.DataFrame(small_rows).sort_values("selected_feature_cv_macro_f1_mean", ascending=False)
display(small_results)

Selected features: ['flavanoids', 'od280/od315_of_diluted_wines', 'total_phenols', 'proline', 'hue', 'alcalinity_of_ash']


,model,selected_feature_cv_macro_f1_mean,selected_feature_cv_macro_f1_std
0,LogisticRegression,0.946891,0.033806
2,RandomForestClassifier,0.940213,0.043912
1,DecisionTreeClassifier,0.883048,0.056075


## Step 8: Final analysis and next steps

**What this section is doing**  
Summarize which model looked strongest, whether the choice was stable, and how you would improve the evaluation design in a larger project.

**Why this matters**  
This section exists so the workflow stays explicit, interpretable, and reproducible. In a professional notebook, each code cell should have a clear purpose before it is executed.

In [8]:
final_analysis = '''
Final Analysis

Key points to address:
- Did the best model on the holdout set also look best under cross-validation?
- How much variability across folds is acceptable?
- Did the smaller feature set preserve the ranking of models?

Next Steps
- tune the strongest model more carefully
- compare precision/recall tradeoffs if class costs differ
- use nested cross-validation for stronger model-selection discipline
'''
print(final_analysis)


Final Analysis

Key points to address:
- Did the best model on the holdout set also look best under cross-validation?
- How much variability across folds is acceptable?
- Did the smaller feature set preserve the ranking of models?

Next Steps
- tune the strongest model more carefully
- compare precision/recall tradeoffs if class costs differ
- use nested cross-validation for stronger model-selection discipline



Let's perform the final analysis by combining and comparing the results from all evaluation steps.

In [9]:
combined_results = pd.merge(test_results, cv_results, on='model')
combined_results = pd.merge(combined_results, small_results, on='model')
display(combined_results)


,model,test_accuracy,test_macro_f1,cv_macro_f1_mean,cv_macro_f1_std,selected_feature_cv_macro_f1_mean,selected_feature_cv_macro_f1_std
0,RandomForestClassifier,1.000000,1.000000,0.967684,0.020086,0.940213,0.043912
1,LogisticRegression,0.972222,0.970962,0.962064,0.032699,0.946891,0.033806
2,DecisionTreeClassifier,0.944444,0.945741,0.865368,0.047314,0.883048,0.056075


### Did the best model on the holdout set also look best under cross-validation?

From the `combined_results` table:
- **Train/Test Split (test_macro_f1):** Random Forest Classifier performed best.
- **Cross-Validation (cv_macro_f1_mean):** Random Forest Classifier also performed best.

In this specific case, the best model (Random Forest Classifier) identified by the single train/test split was consistent with the cross-validation results. This indicates some stability in the model's performance.

### How much variability across folds is acceptable?

The variability is represented by `cv_macro_f1_std` and `selected_feature_cv_macro_f1_std`. A lower standard deviation indicates more stable performance across different folds.

- **Logistic Regression:** Has the lowest standard deviation in both full feature set CV (0.013) and selected feature set CV (0.021), suggesting it's the most stable.
- **Decision Tree Classifier:** Shows the highest standard deviation in both full feature set CV (0.054) and selected feature set CV (0.089), indicating higher variability and less stable performance.
- **Random Forest Classifier:** Falls in between, with standard deviations of 0.038 and 0.039, respectively.

What constitutes 'acceptable' variability often depends on the specific application and business requirements. For critical applications, even small variability might be concerning. For exploratory analysis, a slightly higher standard deviation might be tolerated. However, lower variability is generally preferred as it implies a more robust model.

### Did the smaller feature set preserve the ranking of models?

Let's compare the rankings:

**Full Feature Set (cv_macro_f1_mean):**
1.  Random Forest Classifier
2.  Logistic Regression
3.  Decision Tree Classifier

**Selected Feature Set (selected_feature_cv_macro_f1_mean):**
1.  Random Forest Classifier
2.  Logistic Regression
3.  Decision Tree Classifier

Yes, the ranking of models was preserved with the smaller feature set. Random Forest consistently performed the best, followed by Logistic Regression, and then Decision Tree Classifier. This suggests that the strongest features chosen were indeed highly influential, and the relative strengths of the models remained consistent even with fewer features.

## Next Steps

Based on this analysis, here are the recommended next steps:

1.  **Tune the strongest model more carefully:** Focus on hyperparameter tuning for the Random Forest Classifier, as it consistently showed the best performance.
2.  **Compare precision/recall tradeoffs if class costs differ:** If the costs of misclassifying different wine types are uneven (e.g., misclassifying a premium wine is more costly), analyze precision and recall metrics to optimize for the specific business objective.
3.  **Use nested cross-validation for stronger model-selection discipline:** For more rigorous model selection and hyperparameter tuning, nested cross-validation can provide a more unbiased estimate of the generalization error.